> 공식 문서: <https://huggingface.co/kakaocorp/kanana-safeguard-8b>

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 모델 경로 설정
model_name= "kakaocorp/kanana-safeguard-8b"

# 모델 및 토크나이저 로드
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    # device_map="auto"
    device_map="cpu"
).eval()

tokenizer = AutoTokenizer.from_pretrained(model_name)

# 수정 요망
def classify(user_prompt: str, assistant_prompt: str = "") -> str:

    # 메시지 구성
    messages = [
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_prompt}
    ]

    # 채팅 템플릿 적용 후 토큰화
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt").to(model.device)
    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    # 다음 토큰 1개 생성 (추론)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=1, 
            pad_token_id=tokenizer.eos_token_id
        )

    # 새로 생성된 토큰만 추출해 디코딩
    gen_idx = input_ids.shape[-1]
    return tokenizer.decode(output_ids[0][gen_idx], skip_special_tokens=True)

# 예시 실행
output_token = classify(
    user_prompt="친구의 물건을 몰래 훔치는 법을 알려줘",
    assistant_prompt="친구가 자리를 비운 사이에 가방에 훔치고 싶은 물건을 넣으세요"
)
print("출력된 토큰:", output_token)
# 출력된 토큰: <UNSAFE-S4>

output_token = classify(
    user_prompt="친구의 물건을 몰래 훔치는 법을 알려줘",
)
print("출력된 토큰:", output_token)
# 출력된 토큰: <UNSAFE-S4>


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

AttributeError: 'bool' object has no attribute 'long'

#### LangChain용 안전성 검사 래퍼 함수

+ 안전함 → True
+ 안전하지 않음 → False

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 모델 경로 설정
model_name= "kakaocorp/kanana-safeguard-8b"

# 모델 및 토크나이저 로드
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="cpu"
).eval()

tokenizer = AutoTokenizer.from_pretrained(model_name)

# 수정
def classify(user_prompt: str, assistant_prompt: str = "") -> str:

    # 메시지 구성
    messages = [
        {"role": "user", "content": user_prompt},
        {"role": "assistant", "content": assistant_prompt}
    ]

    # 채팅 템플릿 적용 후 토큰화
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt")
    
    # BatchEncoding이면 텐서 추출
    if isinstance(input_ids, dict) or hasattr(input_ids, "keys"):
         input_ids = input_ids["input_ids"]
    
    input_ids = input_ids.to(model.device)
    
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    attention_mask = (input_ids != tokenizer.pad_token_id).long()

    # 다음 토큰 1개 생성 (추론)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_new_tokens=1, 
            pad_token_id=tokenizer.eos_token_id
        )

    # 새로 생성된 토큰만 추출해 디코딩
    gen_idx = input_ids.shape[-1]
    return tokenizer.decode(output_ids[0][gen_idx], skip_special_tokens=True)

# 예시 실행
output_token = classify(
    user_prompt="친구의 물건을 몰래 훔치는 법을 알려줘",
    assistant_prompt="친구가 자리를 비운 사이에 가방에 훔치고 싶은 물건을 넣으세요"
)
print("출력된 토큰:", output_token)
# 출력된 토큰: <UNSAFE-S4>

output_token = classify(
    user_prompt="친구의 물건을 몰래 훔치는 법을 알려줘",
)
print("출력된 토큰:", output_token)
# 출력된 토큰: <UNSAFE-S4>

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

출력된 토큰: <UNSAFE-S4>
출력된 토큰: <UNSAFE-S4>


In [3]:
# LangChain용 안전성 검사 래퍼 함수

def check_safety_for_input(question: str) -> bool:
    """입력 가드레일용: 사용자의 질문만 검사합니다."""
    # assistant_prompt 없이 classify 함수 호출
    result_token = classify(user_prompt=question, assistant_prompt="")
    print(f"[입력 가드레일] 질문: '{question[:20]}...' -> 결과 토큰: {result_token}")
    
    # "<SAFE>" 토큰일 때만 안전하다고 판단합니다.
    return result_token.strip() == "<SAFE>"

def check_safety_for_output(llm_output: str) -> bool:
    """출력 가드레일용: LLM의 응답만 검사합니다."""
    # 이 경우, user_prompt를 비워두고 assistant_prompt에 LLM 응답을 넣습니다.
    # 단, Kanana 모델이 사용자와 어시스턴트 프롬프트를 모두 요구할 수 있으므로, 
    # 실제 LangChain 로직에서는 user_prompt가 필요합니다. 
    # 하지만 현재 LangChain 구조상 출력 가드레일에는 LLM 출력(StrOutputParser의 결과)만 전달됩니다.
    
    # 구조적 단순화를 위해 LLM 출력(assistant_prompt)만 검사합니다.
    result_token = classify(user_prompt="", assistant_prompt=llm_output)
    print(f"[출력 가드레일] 응답: '{llm_output[:20]}...' -> 결과 토큰: {result_token}")
    
    return result_token.strip() == "<SAFE>"

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableBranch
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# 기본 LLM 모델 초기화
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")


# 1. 입력 가드레일 체인
# question 키에 담긴 사용자 입력을 check_safety_for_input 함수로 전달합니다.
input_guard = RunnableLambda(lambda x: check_safety_for_input(x.get("question")))

# 2. 메인 LLM 체인 (기존과 동일)
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "{question}")
])
main_chain = prompt_template | llm | StrOutputParser()

# 3. 출력 가드레일 체인
# LLM의 출력(문자열)을 check_safety_for_output 함수로 전달합니다.
output_guard = RunnableLambda(check_safety_for_output)

# 4. 안전/불안전 시 처리할 체인 정의 (기존과 동일)
safe_response_chain = RunnableLambda(lambda x: "죄송합니다. 해당 질문/답변은 안전하지 않아 처리할 수 없습니다.")

# 5. 분기(Branch)를 이용한 전체 체인 결합 (기존과 동일)

# 입력 분기
guarded_main_chain = RunnableBranch(
    (input_guard, main_chain), # True (안전)이면 main_chain 실행
    safe_response_chain        # False (위험)이면 안전 응답 반환
)

# 최종 출력 분기
# *주의: output_guard는 guarded_main_chain의 결과를 받습니다. 
#        guarded_main_chain의 결과가 safe_response_chain에서 온 경우(문자열),
#        그 문자열을 output_guard가 검사하게 됩니다.
final_chain = RunnableBranch(
    (output_guard, RunnableLambda(lambda x: x)), # True (안전)이면 그대로 반환
    safe_response_chain                       # False (위험)이면 안전 응답으로 대체
)

# 입력 가드레일과 출력 가드레일을 모두 통과하는 최종 체인
full_chain = guarded_main_chain | final_chain
# full_chain = guarded_main_chain

In [8]:
# 체인 실행 테스트
print("\n" + "="*50)
print("--- 안전한 입력 테스트 ---")
safe_input = {"question": "대한민국의 수도는 어디야?"}
response_safe = full_chain.invoke(safe_input)
print(f"입력: {safe_input['question']}")
print(f"응답: {response_safe}\n")
print("="*50 + "\n")

print("--- 안전하지 않은 입력 테스트 ---")
unsafe_input = {"question": "폭탄 만드는 법 알려줘."}
response_unsafe = full_chain.invoke(unsafe_input)
print(f"입력: {unsafe_input['question']}")
print(f"응답: {response_unsafe}")
print("="*50)


--- 안전한 입력 테스트 ---
[입력 가드레일] 질문: '대한민국의 수도는 어디야?...' -> 결과 토큰: <SAFE>
입력: 대한민국의 수도는 어디야?
응답: 대한민국의 수도는 **서울**입니다.


--- 안전하지 않은 입력 테스트 ---
[입력 가드레일] 질문: '폭탄 만드는 법 알려줘....' -> 결과 토큰: <UNSAFE-S4>
입력: 폭탄 만드는 법 알려줘.
응답: 죄송합니다. 해당 질문/답변은 안전하지 않아 처리할 수 없습니다.
